In [1]:
"""
run_sweep_A.py — Valuation (v) sweep.

Varies product × supplier to get a range of v = cost + internal_cost.
Fixes strategy = cooperative, model = deepseek-reasoner.

Usage:
    python run_sweep_A.py
"""

import sys
from pathlib import Path

# Ensure the project root is importable
# sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from openai import OpenAI

from NegoLib.Entities.Agents import (
    LOCAL_SUPPLIER, NATIONAL_SUPPLIER
)
from NegoLib.Entities.products import (
    SPARKLING_WATER, COLA, ENERGY_DRINK, PROTEIN_BAR,
)

from exp_runner_0412 import (
    batch_random_buyer_exp, build_transitions, save_json,
)

# ---------------------------------------------------------------------------
# Config — edit these
# ---------------------------------------------------------------------------
CLIENT = OpenAI(
    api_key="sk-81e3e6eae33d4486ad4127e0cb3630a6",           # <-- replace
    base_url="https://api.deepseek.com",
)
MODEL = "deepseek-reasoner"
STRATEGY = "cooperative"
N_RUNS = 3
N_ROUNDS = 10
BASE_SEED = 2000

OUTPUT_DIR = Path("results/sweep_A")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ---------------------------------------------------------------------------
# Experiment grid
# ---------------------------------------------------------------------------
PRODUCTS = [
    (SPARKLING_WATER, 24),   # (product, quantity)
    (COLA,            60),
    (PROTEIN_BAR,     40),
]

SUPPLIERS = [
    LOCAL_SUPPLIER,
    NATIONAL_SUPPLIER
]



In [ ]:
all_experiments = []
total = len(PRODUCTS) * len(SUPPLIERS)
idx = 0

for product, qty in PRODUCTS:
    for supplier in SUPPLIERS:
        idx += 1
        v = product.cost + supplier.internal_costs.for_category(product.category)
        tag = f"{product.name[:12]}_{supplier.id}"
        print(f"\n[{idx}/{total}] {tag}  v={v:.2f}")

        experiment = batch_random_buyer_exp(
            client=CLIENT,
            model=MODEL,
            product=product,
            quantity=qty,
            supplier=supplier,
            strategy=STRATEGY,
            n_runs=N_RUNS,
            n_rounds=N_ROUNDS,
            base_seed=BASE_SEED,
            verbose=True,
        )

        save_json(experiment, OUTPUT_DIR / f"A_{tag}.json")
        all_experiments.append(experiment)

# Merge all transitions for joint analysis
all_trans = []
for exp in all_experiments:
    all_trans.extend(build_transitions(exp))

save_json(
    {"sweep": "A", "n_conditions": total, "n_transitions": len(all_trans),
        "transitions": all_trans},
    OUTPUT_DIR / "A_all_transitions.json",
)
print(f"\nSweep A complete.  {len(all_trans)} transitions total.")

# 66 min


[1/6] Sparkling wa_S1  v=0.22

--- Run 1/3 (seed=2000) ---
  R1 BUYER : $0.64  — My offer: Sparkling water (500ml): $0.64 per unit, 24 units.
  R1 SELLER: $0.95  ← "Thank you for your offer. We value building a lasting partnership and are committed to providing quality products. Given..."
         [thinking]: "We are the supplier. The merchant's opening offer is Sparkling water (500ml): $0..."
  R2 BUYER : $1.15  — How about $1.15 per unit for 24 units of Sparkling water (500ml)?
  R2 SELLER: $1.15  ← "I appreciate your revised offer. At $1.15 per unit, you're showing a strong commitment to a fair partnership, and I'm ha..."
         [thinking]: "Okay, the merchant countered with $1.15 per unit for 24 units of sparkling water..."
  R3 BUYER : $0.66  — I'm willing to pay $0.66 per unit for Sparkling water (500ml), 24 units total.
  R3 SELLER: $1.00  ← "Thank you for your offer. However, $0.66 per unit is significantly below the market price of $1.20, and it doesn't refle..."
         [